# Hit Patterns for pT=2.0 GeV Electrons (Python/matplotlib)

Shows truth pT, rec pT, and nHits in the plot title.
- **XY view**: blue=primary (high EDep), red=secondary (low EDep)
- **RZ view**: same colors, detector boundaries shown

Uses pure Python: `uproot` + `matplotlib` (no ROOT required).

In [5]:
import numpy as np
import awkward as ak
import uproot
import os
import matplotlib.pyplot as plt

# Constants
Bz = 3.0
alpha = Bz * 2.99792458e-4  # GeV/(m*T)
TUPLEFILE = "../gsf_flat-e--2.0-85-1.root"

print(f"Tuple file: {TUPLEFILE}")
print(f"Bz = {Bz} T, alpha = {alpha:.6e}")

Tuple file: ../gsf_flat-e--2.0-85-1.root
Bz = 3.0 T, alpha = 8.993774e-04


In [6]:
# Open file and count events
f = uproot.open(TUPLEFILE)
t = f["gsf_tuple"]

n_evt = t.num_entries
print(f"Events: {n_evt}")
print(f"\nBranches in the flat tuple:")
for b in t.keys():
    print(f"  {b}")

Events: 200

Branches in the flat tuple:
  iev
  mc_pdg
  mc_px
  mc_py
  mc_pz
  mc_pT
  mc_p
  mc_eta
  mc_theta
  mc_phi
  mc_vx
  mc_vy
  mc_vz
  lcio_hit_n
  lcio_hit_x
  lcio_hit_y
  lcio_hit_z
  lcio_hit_r
  lcio_hit_edep
  lcio_hit_cellid
  gsf_hit_n
  gsf_hit_x
  gsf_hit_y
  gsf_hit_z
  gsf_hit_r
  gsf_hit_edep
  gsf_hit_cellid
  all_hit_n
  all_hit_x
  all_hit_y
  all_hit_z
  all_hit_r
  all_hit_edep
  all_hit_cellid
  all_hit_det
  lcio_pT
  lcio_p
  lcio_eta
  lcio_theta
  lcio_phi
  lcio_d0
  lcio_z0
  lcio_omega
  lcio_tanl
  lcio_chi2
  lcio_ndf
  lcio_nhits
  lcio_type
  gsf_pT
  gsf_p
  gsf_eta
  gsf_theta
  gsf_phi
  gsf_d0
  gsf_z0
  gsf_omega
  gsf_tanl
  gsf_chi2
  gsf_ndf
  gsf_nhits
  gsf_type
  res_pT_gsf
  res_pT_lcio


In [7]:
# Read all branches from the flat tuple
arrays = t.arrays()

# Build track list from flat tuple
tracks = {"good": [], "bad": []}

for iev in range(n_evt):
    # MC truth
    truth_pt  = arrays["mc_pT"][iev]
    truth_p   = arrays["mc_p"][iev]
    truth_eta = arrays["mc_eta"][iev]
    theta     = arrays["mc_theta"][iev]  # radians
    
    # LCIO
    lcio_pt  = arrays["lcio_pT"][iev]
    lcio_p   = arrays["lcio_p"][iev]
    lcio_chi2 = arrays["lcio_chi2"][iev]
    lcio_ndf  = arrays["lcio_ndf"][iev]
    lcio_nhits = arrays["lcio_nhits"][iev]
    
    # GSF
    gsf_pt   = arrays["gsf_pT"][iev]
    gsf_p    = arrays["gsf_p"][iev]
    gsf_chi2 = arrays["gsf_chi2"][iev]
    gsf_ndf  = arrays["gsf_ndf"][iev]
    
    # Resolution
    res_gsf  = arrays["res_pT_gsf"][iev]
    res_lcio = arrays["res_pT_lcio"][iev]
    
    # Per-hit data from LCIO track
    lcio_hit_n = arrays["lcio_hit_n"][iev]
    lcio_hit_x = arrays["lcio_hit_x"][iev] if lcio_hit_n > 0 else []
    lcio_hit_y = arrays["lcio_hit_y"][iev] if lcio_hit_n > 0 else []
    lcio_hit_z = arrays["lcio_hit_z"][iev] if lcio_hit_n > 0 else []
    lcio_hit_r = arrays["lcio_hit_r"][iev] if lcio_hit_n > 0 else []
    lcio_hit_edep = arrays["lcio_hit_edep"][iev] if lcio_hit_n > 0 else []
    
    if truth_pt <= 0 or gsf_pt <= 0:
        continue
    
    # Build per-hit dict list (same format as old notebook expects)
    hits = []
    for j in range(lcio_hit_n):
        hits.append({
            "x": lcio_hit_x[j],
            "y": lcio_hit_y[j],
            "z": lcio_hit_z[j],
            "r": lcio_hit_r[j],
            "edep": lcio_hit_edep[j],
        })
    
    tdata = {
        "iev": iev,
        "truth_pt": truth_pt,
        "truth_p": truth_p,
        "lcio_pt": lcio_pt,
        "lcio_p": lcio_p,
        "gsf_pt": gsf_pt,
        "gsf_p": gsf_p,
        "rec_pt": gsf_pt,       # use GSF as "reconstructed" for plot title
        "res": res_gsf * 100,   # percent
        "trk_hits": lcio_hit_n,
        "theta": np.degrees(theta),
        "hits": hits,
    }
    
    if abs(res_gsf) < 0.1:
        tracks["good"].append(tdata)
    else:
        tracks["bad"].append(tdata)

print(f"Good tracks: {len(tracks['good'])}")
print(f"Bad tracks:  {len(tracks['bad'])}")

Good tracks: 183
Bad tracks:  17


In [8]:
# Select top 5 good and top 5 bad tracks
good5 = sorted(tracks["good"], key=lambda t: abs(t["res"]))[:5]
bad5  = sorted(tracks["bad"], key=lambda t: abs(t["res"]), reverse=True)[:5]

outdir = "hit_plots_2GeV_85deg"
os.makedirs(outdir, exist_ok=True)

DET_RINGS = [14, 50, 150, 350, 600, 1810]

print(f"Output directory: {outdir}/")
print(f"Detector rings (mm): {DET_RINGS}")

Output directory: hit_plots_2GeV_85deg/
Detector rings (mm): [14, 50, 150, 350, 600, 1810]


In [9]:
def draw_rings(ax, xy=True):
    """Draw detector boundary circles on XY view or lines on RZ view."""
    if xy:
        for r in DET_RINGS:
            circle = plt.Circle((0, 0), r, fill=False, color="gray",
                                linestyle="--", linewidth=0.8, alpha=0.5)
            ax.add_artist(circle)
    else:
        for r in DET_RINGS:
            ax.axhline(y=r, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.axvline(x=0, color="gray", linestyle=":", linewidth=0.8, alpha=0.4)


def make_title(t, cat):
    """Build a descriptive title string for a track."""
    return "%s  evt %d  truth=%.3f  rec=%.3f  res=%.1f%%  nHits=%d  theta=%.0f deg" % (
        cat, t["iev"], t["truth_pt"], t["rec_pt"], t["res"], t["trk_hits"], t["theta"]
    )

## Individual XY Hit Pattern Plots

In [ ]:
# Generate XY PNGs — pure matplotlib
for idx, t in enumerate(good5 + bad5):
    cat = "GOOD" if idx < 5 else "BAD"
    title = make_title(t, cat)
    safe = title.replace(" ", "_").replace("=", "").replace("%", "pct").replace(".", "_")
    fname = "%s/hit_xy_%d_%s.png" % (outdir, idx, safe[:80])

    xs = [h["x"] for h in t["hits"]]
    ys = [h["y"] for h in t["hits"]]
    eds = [h["edep"] for h in t["hits"]]

    rmax = max([abs(x) + abs(y) for x, y in zip(xs, ys)]) * 1.15 if xs else 2000
    limit = max(rmax, 50)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect("equal")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.set_title(title, fontsize=10)
    draw_rings(ax, xy=True)

    # Separate high-EDep (primary) and low-EDep (secondary) hits
    hi_x, hi_y = [], []
    lo_x, lo_y = [], []
    for x, y, ed in zip(xs, ys, eds):
        if ed > 1e-4:
            hi_x.append(x); hi_y.append(y)
        else:
            lo_x.append(x); lo_y.append(y)

    if hi_x:
        ax.scatter(hi_x, hi_y, s=3, c="blue", marker=".", label="EDep > 1e-4")
    if lo_x:
        ax.scatter(lo_x, lo_y, s=6, c="red", marker=".", label="EDep ≤ 1e-4")

    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    print("  %s" % fname)

print("\nXY plots done.")

## Individual RZ Hit Pattern Plots

In [ ]:
# Generate RZ PNGs — pure matplotlib
for idx, t in enumerate(good5 + bad5):
    cat = "GOOD" if idx < 5 else "BAD"
    title = make_title(t, cat)
    safe = title.replace(" ", "_").replace("=", "").replace("%", "pct").replace(".", "_")
    fname = "%s/hit_rz_%d_%s.png" % (outdir, idx, safe[:80])

    xs = [h["x"] for h in t["hits"]]
    ys = [h["y"] for h in t["hits"]]
    zs = [h["z"] for h in t["hits"]]
    rs = [np.hypot(x, y) for x, y in zip(xs, ys)]
    eds = [h["edep"] for h in t["hits"]]

    zmin = min(zs) * 1.1 if zs else -2000
    zmax = max(zs) * 1.1 if zs else 2000
    rmax = max(rs) * 1.2 if rs else 2000
    rmax = max(rmax, 50)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.set_xlim(zmin, zmax)
    ax.set_ylim(0, rmax)
    ax.set_xlabel("z [mm]")
    ax.set_ylabel("r [mm]")
    ax.set_title(title, fontsize=10)
    draw_rings(ax, xy=False)

    hi_z, hi_r = [], []
    lo_z, lo_r = [], []
    for z, r, ed in zip(zs, rs, eds):
        if ed > 1e-4:
            hi_z.append(z); hi_r.append(r)
        else:
            lo_z.append(z); lo_r.append(r)

    if hi_z:
        ax.scatter(hi_z, hi_r, s=3, c="blue", marker=".", label="EDep > 1e-4")
    if lo_z:
        ax.scatter(lo_z, lo_r, s=6, c="red", marker=".", label="EDep ≤ 1e-4")

    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    print("  %s" % fname)

print("\nRZ plots done.")

## Summary Table

In [ ]:
# Print summary table
print("=" * 110)
for tag, list_ in [("GOOD", good5), ("BAD", bad5)]:
    print(f"\n{tag} (pT=2.0 GeV, theta=85 deg):")
    for t in list_:
        # TPC spread: evaluate scatter of r vs z in TPC-like region (|z| > 300 mm)
        tpc_zs = np.array([h["z"] for h in t["hits"] if abs(h["z"]) > 300])
        tpc_rs = np.array([np.hypot(h["x"], h["y"]) for h in t["hits"] if abs(h["z"]) > 300])
        spread = 0
        if len(tpc_zs) > 10:
            cfs = np.polyfit(tpc_zs, tpc_rs, 1)
            spread = np.std(tpc_rs - np.polyval(cfs, tpc_zs))
        print("  evt %3d  truth=%.3f  rec=%.3f  res=%7.2f%%  nHits=%4d  TPC_r_z_spread=%.0fmm" %
              (t["iev"], t["truth_pt"], t["rec_pt"], t["res"], t["trk_hits"], spread))

print(f"\nDone. 20 files in {outdir}/")